# FOVL Assignment — Multilingual Word Embeddings for Indian Languages

**Approach:** FastText (skip-gram with character n-grams, gensim) trained **from scratch**, 300-d, **only on IndicCorpV2**.
Out-of-vocabulary words get vectors composed from their character n-grams (subword strategy), so no word pair is ever dropped.

**Structure**
* **Part A** – setup, text normalisation, benchmark loader (always run)
* **Part B** – load IndicCorpV2, run experiments, pick the final models, train extra languages
* **Part C** – inference API for the private test: `get_embedding(word, lang)` / `embed_pairs(pairs, lang)`
* **Part D** – export (`.npy`, vocab files), reproduction check, t-SNE, analysis tables, OOV stats, zip packaging

If `outputs/models/` already exists (e.g. after unzipping the submission) you can skip Part B and run A → C.

**Rules followed:** no pre-trained vectors, no external corpora; benchmark human scores are never used to train vectors
(they are used only to *report* and to choose among a small, pre-declared set of configurations).

## Part A — Setup

In [ ]:
import subprocess, sys
def _pip(*pkgs):
    try: subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
    except Exception as e: print("pip skipped:", e)
_pip("gensim", "datasets", "regex", "scikit-learn", "scipy", "matplotlib", "pandas")

In [ ]:
import os, re, json, time, random, shutil, zipfile, unicodedata, collections
from pathlib import Path
import numpy as np, pandas as pd, regex
from scipy.stats import spearmanr
from gensim.models import FastText
from gensim.models.fasttext import FastTextKeyedVectors

# ----------------------------- configuration ---------------------------------
SMOKE      = os.environ.get("FOVL_SMOKE", "0") == "1"   # synthetic stand-in corpus, for mechanics testing ONLY
QUICK      = os.environ.get("FOVL_QUICK", "0") == "1"   # fewer/lighter experiments
HF_DATASET = "rony000013/indiccorpV2-5k"
BENCH_REPO = "https://github.com/syedsarfarazakhtar/Word-Similarity-Datasets-for-Indian-Languages"
BENCH_DIR  = Path(os.environ.get("FOVL_BENCH", "Word-Similarity-Datasets-for-Indian-Languages"))
OUT        = Path("outputs_smoke" if SMOKE else "outputs")
WORK       = OUT / "work"
MODELS     = OUT / "models"
DIM, SEED  = 300, 42
WORKERS    = max(1, os.cpu_count() or 1)
MAX_SENT_LEN = 1000
for d in (OUT, WORK, MODELS, OUT/"embeddings", OUT/"vocab", OUT/"results", OUT/"figures"): d.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print("SMOKE" if SMOKE else "REAL", "mode | output dir:", OUT, "| workers:", WORKERS)

In [ ]:
# ----------------------------- text normalisation ----------------------------
_STRIP = dict.fromkeys(map(ord, "‌‍‎‏﻿­"), None)      # ZWNJ/ZWJ/marks/BOM/soft hyphen
_URDU  = {ord("ي"): "ی", ord("ى"): "ی", ord("ك"): "ک", ord("ە"): "ہ"}               # Arabic -> Urdu letter forms
_TOKEN = regex.compile(r"[\p{L}\p{M}\p{N}]+")        # letters + combining marks (matras!) + numbers
_SENT  = regex.compile(r"[\n\r।॥۔؟\.\?!;]+")

def normalize(s: str) -> str:
    return unicodedata.normalize("NFC", s).translate(_STRIP).translate(_URDU).lower()

def tokenize(s: str):
    return _TOKEN.findall(normalize(s))

def text_to_sentences(text: str):
    out = []
    for part in _SENT.split(text or ""):
        toks = tokenize(part)
        if len(toks) >= 2: out.append(toks[:MAX_SENT_LEN])
    return out

def char_ngrams(w, n1, n2):
    w = f"<{w}>"
    return {w[i:i+n] for n in range(n1, n2+1) for i in range(len(w)-n+1)}

# canonical language names (+ aliases seen in dataset configs / columns / codes)
_ALIASES = {
 "Hindi":["hi","hin","hindi","hin_deva"], "Gujarati":["gu","guj","gujarati","guj_gujr"],
 "Marathi":["mr","mar","marathi","mar_deva"], "Punjabi":["pa","pan","punjabi","panjabi","pan_guru"],
 "Tamil":["ta","tam","tamil","tam_taml"], "Telugu":["te","tel","telugu","tel_telu"],
 "Urdu":["ur","urd","urdu","urd_arab"], "Bengali":["bn","ben","bengali","bangla","ben_beng"],
 "Kannada":["kn","kan","kannada","kan_knda"], "Malayalam":["ml","mal","malayalam","mal_mlym"],
 "Odia":["or","ory","odia","oriya","ory_orya"], "Assamese":["as","asm","assamese","asm_beng"],
 "Nepali":["ne","npi","nepali","npi_deva"], "Sanskrit":["sa","san","sanskrit","san_deva"],
 "Sindhi":["sd","snd","sindhi"], "Konkani":["kok","konkani"], "Maithili":["mai","maithili"],
 "Dogri":["doi","dogri"], "Kashmiri":["ks","kas","kashmiri"], "Bodo":["brx","bodo"],
 "Manipuri":["mni","manipuri","meitei"], "Santali":["sat","santali"], "English":["en","eng","english"],
}
_LOOKUP = {a: k for k, v in _ALIASES.items() for a in v}
def canon_lang(x):
    if x is None: return None
    s = str(x).strip().lower().replace("-", "_")
    if s in _LOOKUP: return _LOOKUP[s]
    if s.split("_")[0] in _LOOKUP: return _LOOKUP[s.split("_")[0]]
    return None

def script_lang(text):               # last-resort guess from the dominant script (ambiguous for Devanagari)
    c = collections.Counter()
    for ch in text[:300]:
        try: c[unicodedata.name(ch).split()[0]] += 1
        except ValueError: pass
    top = c.most_common(1)[0][0] if c else ""
    return {"GUJARATI":"Gujarati","GURMUKHI":"Punjabi","TAMIL":"Tamil","TELUGU":"Telugu","ARABIC":"Urdu",
            "BENGALI":"Bengali","KANNADA":"Kannada","MALAYALAM":"Malayalam","ORIYA":"Odia"}.get(top)

In [ ]:
# ----------------------------- benchmark (Akhtar et al., 2017) ---------------
if not BENCH_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", BENCH_REPO, str(BENCH_DIR)], check=False)

def read_pairs(path):
    # Returns ALL pairs in file order as (word1, word2, score_or_None).
    # Hindi is comma separated, the others tab separated (CRLF, sometimes a trailing tab / an extra 4th column).
    # NOTE: the public Marathi file contains pairs with an EMPTY score -> kept (they still need embeddings), score=None.
    rows = []
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip("\r\n")
            if not line.strip(): continue
            parts = [p.strip() for p in (line.split("\t") if "\t" in line else line.split(","))]
            if len(parts) < 2: continue
            score = None
            if len(parts) >= 3 and parts[2] != "":
                try: score = float(parts[2])
                except ValueError: score = None
            rows.append((parts[0], parts[1], score))
    return rows

BENCH_ALL = {}
for p in sorted(BENCH_DIR.glob("*-WS.txt")):
    BENCH_ALL[p.name.split("-")[0]] = read_pairs(p)
BENCH_LANGS = list(BENCH_ALL)
BENCH = {l: [r for r in rows if r[2] is not None] for l, rows in BENCH_ALL.items()}      # scored pairs: used for Spearman
SCORED_IDX = {l: [i for i, r in enumerate(rows) if r[2] is not None] for l, rows in BENCH_ALL.items()}
print("all pairs   :", {k: len(v) for k, v in BENCH_ALL.items()})
print("scored pairs:", {k: len(v) for k, v in BENCH.items()}, "(only these enter Spearman)")
print("multi-word entries:", {k: sum((len(tokenize(a)) > 1) + (len(tokenize(b)) > 1) for a, b, _ in v) for k, v in BENCH_ALL.items()})

In [ ]:
# ----------------------------- embedder (OOV via subwords) -------------------
class Embedder:
    # Wraps a FastTextKeyedVectors object.
    # * in-vocabulary word -> its trained vector;  OOV word -> mean of its character n-gram vectors (FastText)
    # * multi-token entries (e.g. Urdu 'کی بورڈ') -> mean of token vectors
    # * optional unsupervised 'all-but-the-top' post-processing (mean removal + top-d PCs), fitted on corpus vectors only
    def __init__(self, wv, mu=None, pcs=None, post_d=0):
        self.wv, self.mu, self.pcs, self.post_d = wv, mu, pcs, post_d
    def _tok(self, t):
        try: return np.asarray(self.wv[t], dtype=np.float32)
        except KeyError: return np.zeros(DIM, np.float32)
    def vec(self, text, post_d=None):
        d = self.post_d if post_d is None else post_d
        toks = tokenize(text) or [normalize(text)]
        v = np.mean([self._tok(t) for t in toks], axis=0)
        if d > 0 and self.mu is not None:
            v = v - self.mu
            for u in self.pcs[:d]: v = v - float(v @ u) * u
        return v.astype(np.float32)
    def is_oov(self, text):
        toks = tokenize(text) or [normalize(text)]
        return any(t not in self.wv.key_to_index for t in toks)

def fit_post(wv, topn=30000, k=3):
    X = np.asarray(wv.vectors[:min(topn, len(wv.index_to_key))], dtype=np.float32)
    mu = X.mean(0); Xc = X - mu
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return mu.astype(np.float32), Vt[:k].astype(np.float32)

def cos(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return 0.0 if na < 1e-9 or nb < 1e-9 else float(a @ b / (na * nb))

def evaluate(emb, pairs, post_d=None):
    sims = [cos(emb.vec(a, post_d), emb.vec(b, post_d)) for a, b, _ in pairs]
    gold = [s for _, _, s in pairs]
    r = spearmanr(sims, gold)[0]
    return (0.0 if np.isnan(r) else float(r)), sims

def coverage(emb, pairs, n1=3, n2=6):
    words  = sorted({w for a, b, _ in pairs for w in (a, b)})
    toks   = sorted({t for w in words for t in (tokenize(w) or [normalize(w)])})
    oov    = [t for t in toks if t not in emb.wv.key_to_index]
    vocab_ng = set()
    for w in emb.wv.index_to_key[:200000]: vocab_ng |= char_ngrams(w, n1, n2)
    ng_cov = [len(char_ngrams(t, n1, n2) & vocab_ng) / max(1, len(char_ngrams(t, n1, n2))) for t in oov]
    return dict(unique_words=len(words), unique_tokens=len(toks), in_vocab=len(toks)-len(oov), oov_tokens=len(oov),
                oov_token_pct=round(100*len(oov)/max(1,len(toks)), 2),
                pairs=len(pairs), pairs_with_oov=sum(emb.is_oov(a) or emb.is_oov(b) for a, b, _ in pairs),
                oov_ngram_coverage_pct=round(100*float(np.mean(ng_cov)), 1) if ng_cov else None)

## Part B — Corpus, experiments, final models

In [ ]:
# ----------------------------- IndicCorpV2 loader ----------------------------
# The dataset schema is detected automatically (language column / per-language configs / per-language splits /
# one column per language).  The schema is printed — if the detection is wrong, adjust ONLY this cell.
TEXT_COL_CANDIDATES = ["text", "sentence", "content", "raw_text", "document", "data"]
LANG_COL_CANDIDATES = ["lang", "language", "Language", "lang_code", "language_code", "iso", "src_lang"]

def load_corpus_real():
    from datasets import load_dataset, get_dataset_config_names
    try: cfgs = [c for c in get_dataset_config_names(HF_DATASET)] or [None]
    except Exception: cfgs = [None]
    corpus_text = collections.defaultdict(list)
    for cfg in cfgs:
        ds = load_dataset(HF_DATASET) if cfg in (None, "default") else load_dataset(HF_DATASET, cfg)
        items = ds.items() if hasattr(ds, "items") else [("train", ds)]
        for split, d in items:
            cols = d.column_names
            print(f"[schema] config={cfg} split={split} rows={len(d)} columns={cols} features={ {c: str(d.features[c]) for c in cols} }")
            lang_col = next((c for c in LANG_COL_CANDIDATES if c in cols), None)
            text_col = next((c for c in TEXT_COL_CANDIDATES if c in cols), None)
            fixed = canon_lang(cfg) or canon_lang(split)
            if text_col is None:                       # wide format: one column per language?
                wide = {c: canon_lang(c) for c in cols if canon_lang(c)}
                if wide:
                    for c, l in wide.items(): corpus_text[l] += [t for t in d[c] if isinstance(t, str)]
                    continue
                strcols = [c for c in cols if "string" in str(d.features[c]).lower()]
                text_col = strcols[0] if strcols else None
            if text_col is None: raise RuntimeError(f"cannot find a text column in {cols}")
            texts = d[text_col]
            langs = d[lang_col] if lang_col else None
            for i, t in enumerate(texts):
                if not isinstance(t, str): continue
                l = canon_lang(langs[i]) if langs is not None else fixed
                l = l or fixed or script_lang(t) or "Unknown"
                corpus_text[l].append(t)
    return corpus_text

def load_corpus_smoke():
    # NOT real data. Tiny synthetic corpus built from benchmark words, only to test the notebook mechanics offline.
    rng = random.Random(SEED); out = collections.defaultdict(list)
    for lang, pairs in BENCH.items():
        words = sorted({w for a, b, _ in pairs for w in (a, b)})
        for _ in range(3000):
            a, b, s = rng.choice(pairs)
            noise = rng.sample(words, 4)
            out[lang].append(" ".join([a, b] + noise) + " .")
    return out

t0 = time.time()
corpus_text = load_corpus_smoke() if SMOKE else load_corpus_real()
CORPUS = {l: [s for t in texts for s in text_to_sentences(t)] for l, texts in corpus_text.items()}
del corpus_text
stats = pd.DataFrame([{"language": l, "sentences": len(s), "tokens": sum(map(len, s)),
                       "types": len({t for x in s for t in x})} for l, s in CORPUS.items()]).sort_values("tokens", ascending=False)
stats.to_csv(OUT/"results"/"corpus_stats.csv", index=False)
print(stats.to_string(index=False)); print(f"loaded in {time.time()-t0:.0f}s")
missing = [l for l in BENCH_LANGS if l not in CORPUS or len(CORPUS[l]) == 0]
assert not missing, f"No training text found for benchmark languages: {missing} -> fix language detection in the loader cell"

In [ ]:
# ----------------------------- training / experiments ------------------------
BASE = dict(sg=1, window=5, min_count=2, negative=10, sample=1e-4, min_n=3, max_n=6,
            bucket=100_000, epochs=10, alpha=0.025, min_alpha=0.0001)
EXPERIMENTS = {          # small, pre-declared grid (each trained mono-lingually per language AND as one multilingual model)
    "sg_base":     {},
    "cbow":        dict(sg=0, alpha=0.05),
    "sg_ngram2-5": dict(min_n=2, max_n=5),
    "sg_ep20":     dict(epochs=20),
    "sg_win8_ep20": dict(window=8, epochs=20),
    "sg_ngram2-6_ep20": dict(min_n=2, max_n=6, epochs=20),
}
if QUICK: EXPERIMENTS = {k: EXPERIMENTS[k] for k in ("sg_base", "cbow")}
POST_DS = [0, 1, 2, 3]
SMALL_BUCKET = SMOKE
def cfg_of(over):
    c = {**BASE, **over}
    if SMALL_BUCKET: c["bucket"] = 20_000
    if SMOKE: c["min_count"] = 1; c["epochs"] = min(c["epochs"], 5)
    return c

def train_ft(sentences, cfg):
    m = FastText(vector_size=DIM, window=cfg["window"], min_count=cfg["min_count"], sg=cfg["sg"], hs=0,
                 negative=cfg["negative"], sample=cfg["sample"], min_n=cfg["min_n"], max_n=cfg["max_n"],
                 bucket=cfg["bucket"], alpha=cfg["alpha"], min_alpha=cfg["min_alpha"], workers=WORKERS, seed=SEED)
    m.build_vocab(corpus_iterable=sentences)
    m.train(corpus_iterable=sentences, total_examples=m.corpus_count, epochs=cfg["epochs"])
    wv = m.wv; del m
    return wv

def eval_models(wvs, post_ds=POST_DS):
    # wvs: {lang: wv} -> {post_d: {lang: rho}}
    res = {d: {} for d in post_ds}
    for lang in BENCH_LANGS:
        wv = wvs[lang]; mu, pcs = fit_post(wv)
        e = Embedder(wv, mu, pcs)
        for d in post_ds: res[d][lang] = evaluate(e, BENCH[lang], d)[0]
    return res

def save_set(wvs, scope, folder, cfg, post_d, name):
    folder = Path(folder)
    if folder.exists(): shutil.rmtree(folder)
    folder.mkdir(parents=True)
    meta = {"scope": scope, "config_name": name, "config": cfg, "post_d": post_d, "languages": {}}
    for lang, wv in wvs.items():
        sub = folder / lang; sub.mkdir()
        wv.save(str(sub / "wv.kv"))
        mu, pcs = fit_post(wv); np.savez(sub / "post.npz", mu=mu, pcs=pcs)
        meta["languages"][lang] = lang
    json.dump(meta, open(folder / "meta.json", "w"), indent=1)

RES_PATH = OUT / "results" / "experiments.csv"
BEST = {"mono": {"macro": -9}, "multi": {"macro": -9}}
if RES_PATH.exists() and (WORK/"best_mono"/"meta.json").exists() and (WORK/"best_multi"/"meta.json").exists() and os.environ.get("FOVL_RERUN") != "1":
    print("Experiments already done -> reusing", RES_PATH)
    table = pd.read_csv(RES_PATH)
else:
    rows = []
    multi_sents = [s for l in CORPUS for s in CORPUS[l]]
    random.Random(SEED).shuffle(multi_sents)
    for name, over in EXPERIMENTS.items():
        cfg = cfg_of(over)
        # ---- monolingual: one model per benchmark language
        t0 = time.time()
        mono = {l: train_ft(CORPUS[l], cfg) for l in BENCH_LANGS}
        res = eval_models(mono)
        for d, per in res.items():
            macro = float(np.mean(list(per.values())))
            rows.append(dict(config=name, scope="mono", post_d=d, macro=macro, **per))
            if macro > BEST["mono"]["macro"]:
                BEST["mono"] = dict(macro=macro, name=name, cfg=cfg, post_d=d)
        print(f"[{name}] mono  macro(post=0..3) = {[round(float(np.mean(list(res[d].values()))),4) for d in POST_DS]}  ({time.time()-t0:.0f}s)")
        if BEST["mono"]["name"] == name: save_set(mono, "mono", WORK/"best_mono", cfg, BEST["mono"]["post_d"], name)
        del mono
        # ---- multilingual: one model on all languages together; evaluated for every benchmark language
        t0 = time.time()
        wv = train_ft(multi_sents, cfg)
        res = eval_models({l: wv for l in BENCH_LANGS})
        for d, per in res.items():
            macro = float(np.mean(list(per.values())))
            rows.append(dict(config=name, scope="multi", post_d=d, macro=macro, **per))
            if macro > BEST["multi"]["macro"]:
                BEST["multi"] = dict(macro=macro, name=name, cfg=cfg, post_d=d)
        print(f"[{name}] multi macro(post=0..3) = {[round(float(np.mean(list(res[d].values()))),4) for d in POST_DS]}  ({time.time()-t0:.0f}s)")
        if BEST["multi"]["name"] == name: save_set({"multilingual": wv}, "multi", WORK/"best_multi", cfg, BEST["multi"]["post_d"], name)
        del wv
        table = pd.DataFrame(rows); table.to_csv(RES_PATH, index=False)
table = table.sort_values("macro", ascending=False)
table.to_csv(RES_PATH, index=False)
print(table.round(4).head(12).to_string(index=False))

In [ ]:
# ----------------------------- choose final models ---------------------------
mono_meta  = json.load(open(WORK/"best_mono"/"meta.json"))
multi_meta = json.load(open(WORK/"best_multi"/"meta.json"))
def best_macro(scope):
    return float(table[table.scope == scope].macro.max())
primary = "mono" if best_macro("mono") >= best_macro("multi") else "multi"
print(f"best mono  : {mono_meta['config_name']} post_d={mono_meta['post_d']}  macro={best_macro('mono'):.4f}")
print(f"best multi : {multi_meta['config_name']} post_d={multi_meta['post_d']}  macro={best_macro('multi'):.4f}")
print("PRIMARY SCOPE =", primary)

# final models dir: copy best sets; additionally train monolingual models for every OTHER language in the corpus
if MODELS.exists(): shutil.rmtree(MODELS)
MODELS.mkdir(parents=True)
shutil.copytree(WORK/"best_mono",  MODELS/"mono")
shutil.copytree(WORK/"best_multi", MODELS/"multi")
extra = [l for l in CORPUS if l not in BENCH_LANGS and l != "Unknown" and len(CORPUS[l]) >= 200]
for l in extra:
    wv = train_ft(CORPUS[l], mono_meta["config"])
    sub = MODELS/"mono"/l; sub.mkdir(); wv.save(str(sub/"wv.kv"))
    mu, pcs = fit_post(wv); np.savez(sub/"post.npz", mu=mu, pcs=pcs)
    mono_meta["languages"][l] = l
json.dump(mono_meta, open(MODELS/"mono"/"meta.json", "w"), indent=1)
json.dump({"primary": primary, "dim": DIM, "bench_languages": BENCH_LANGS, "extra_languages": extra}, open(MODELS/"config.json", "w"), indent=1)
print("extra monolingual models:", extra)

## Part C — Inference API (used for the private test)

Everything needed to load the saved models and embed **arbitrary words** (including OOV) lives in this cell.
It only depends on Part A (normalisation/`Embedder`/`read_pairs`) and the files in `outputs/models/`.

```python
get_embedding("किताब", "Hindi")                    # -> (300,) float32
embed_pairs([("w1", "w2"), ...], "Tamil")          # -> (N, 2, 300) float32, order preserved
embed_pair_file("path/to/Telugu-WS.txt", "Telugu") # reads a pair file (tab/comma separated) -> (N, 2, 300)
```
Language names are matched case-insensitively and accept ISO codes (`hi`, `hin`, ...). A language without its own model
falls back to the multilingual model.

In [ ]:
class EmbeddingService:
    def __init__(self, models_dir=MODELS):
        models_dir = Path(models_dir)
        self.cfg = json.load(open(models_dir/"config.json"))
        self.sets = {}
        for scope in ("mono", "multi"):
            meta_p = models_dir/scope/"meta.json"
            if not meta_p.exists(): continue
            meta = json.load(open(meta_p)); self.sets[scope] = {"meta": meta, "emb": {}}
            for lang in meta["languages"]:
                sub = models_dir/scope/lang
                wv = FastTextKeyedVectors.load(str(sub/"wv.kv"))
                p = np.load(sub/"post.npz")
                self.sets[scope]["emb"][lang] = Embedder(wv, p["mu"], p["pcs"], meta["post_d"])
        self.primary = self.cfg["primary"]
    def _pick(self, lang):
        name = canon_lang(lang) or str(lang).strip().title()
        order = [self.primary] + [s for s in self.sets if s != self.primary]
        for scope in order:
            emb = self.sets[scope]["emb"]
            if name in emb: return emb[name]
            if "multilingual" in emb: return emb["multilingual"]   # multilingual model also covers unseen languages
        raise KeyError(f"no model for language {lang!r}")
    def get_embedding(self, word, lang):
        return self._pick(lang).vec(word)
    def embed_pairs(self, pairs, lang):
        e = self._pick(lang)
        out = np.zeros((len(pairs), 2, DIM), np.float32)
        for i, p in enumerate(pairs):
            out[i, 0], out[i, 1] = e.vec(p[0]), e.vec(p[1])
        return out
    def embed_pair_file(self, path, lang):
        return self.embed_pairs([(a, b) for a, b, _ in read_pairs(path)], lang)

SERVICE = EmbeddingService()
get_embedding = SERVICE.get_embedding
embed_pairs   = SERVICE.embed_pairs
embed_pair_file = SERVICE.embed_pair_file
v = get_embedding(BENCH["Hindi"][0][0], "Hindi"); w = get_embedding("एकदमअनदेखाशब्द", "hi")   # last one is a made-up OOV word
print(v.shape, w.shape, "norm(OOV vec) =", round(float(np.linalg.norm(w)), 3))

## Part D — Export, verification, t-SNE, analysis

In [ ]:
# ----------------------------- results via the *saved* models ----------------
final_rows, cov_rows, SIMS = [], [], {}
for lang in BENCH_LANGS:
    e = SERVICE._pick(lang)
    rho, sims = evaluate(e, BENCH[lang]); SIMS[lang] = sims
    final_rows.append({"language": lang, "scored_pairs": len(BENCH[lang]), "all_pairs": len(BENCH_ALL[lang]), "spearman": round(rho, 4)})
    cov_rows.append({"language": lang, **coverage(e, BENCH_ALL[lang], e.wv.min_n, e.wv.max_n)})
final = pd.DataFrame(final_rows)
macro = float(final.spearman.mean())
final.loc[len(final)] = {"language": "MACRO-AVG", "scored_pairs": int(final.scored_pairs.sum()), "all_pairs": int(final.all_pairs.sum()), "spearman": round(macro, 4)}
final.to_csv(OUT/"results"/"final_spearman.csv", index=False)
cov = pd.DataFrame(cov_rows); cov.to_csv(OUT/"results"/"oov_coverage.csv", index=False)
print(final.to_string(index=False)); print(); print(cov.to_string(index=False))

In [ ]:
# ----------------------------- export .npy + vocab files ---------------------
for lang in BENCH_LANGS:
    e = SERVICE._pick(lang)
    pairs = BENCH_ALL[lang]                                                    # ALL pairs, original file order
    arr = SERVICE.embed_pairs([(a, b) for a, b, _ in pairs], lang)             # (N, 2, 300), original order
    assert arr.shape == (len(pairs), 2, DIM) and arr.dtype == np.float32 and np.isfinite(arr).all()
    np.save(OUT/"embeddings"/f"{lang}_embeddings.npy", arr)
    words = sorted({w for a, b, _ in pairs for w in (a, b)})                    # vocabulary file: word -> row in *_word_vectors.npy
    np.save(OUT/"vocab"/f"{lang}_word_vectors.npy", np.stack([e.vec(w) for w in words]).astype(np.float32))
    json.dump({"language": lang, "dim": DIM, "words": {w: i for i, w in enumerate(words)}},
              open(OUT/"vocab"/f"{lang}_vocab.json", "w", encoding="utf-8"), ensure_ascii=False, indent=0)
    pd.DataFrame({"pair_index": range(len(pairs)), "word1": [p[0] for p in pairs], "word2": [p[1] for p in pairs],
                  "human_score": [p[2] for p in pairs],
                  "word1_oov": [e.is_oov(p[0]) for p in pairs], "word2_oov": [e.is_oov(p[1]) for p in pairs],
                  "cosine": [cos(x[0], x[1]) for x in arr]}).to_csv(OUT/"vocab"/f"{lang}_pairs.tsv", sep="\t", index=False)
print(sorted(p.name for p in (OUT/"embeddings").iterdir()))

# ---- reproduction check: re-load the saved arrays and recompute Spearman from them only
chk = []
for lang in BENCH_LANGS:
    arr = np.load(OUT/"embeddings"/f"{lang}_embeddings.npy")
    assert arr.shape == (len(BENCH_ALL[lang]), 2, DIM)
    sims = [cos(arr[i][0], arr[i][1]) for i in SCORED_IDX[lang]]
    chk.append(float(spearmanr(sims, [s for _, _, s in BENCH[lang]])[0]))
print("Spearman recomputed from saved .npy:", dict(zip(BENCH_LANGS, np.round(chk, 4))), "| macro =", round(float(np.mean(chk)), 4))
assert abs(float(np.mean(chk)) - macro) < 1e-4

In [ ]:
# ----------------------------- t-SNE (100 words / language) ------------------
# Points are labelled with numbers; word <-> number tables are saved (matplotlib cannot shape Indic/Arabic text reliably).
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def pick_words(lang, e, n=100):
    rnd = random.Random(SEED)
    bw = sorted({w for a, b, _ in BENCH_ALL[lang] for w in (a, b)})
    rnd.shuffle(bw); chosen = [(w, "benchmark") for w in bw[:n]]
    if len(chosen) < n:
        have = {w for w, _ in chosen}
        for w in e.wv.index_to_key:
            if w not in have and len(w) > 1: chosen.append((w, "frequent")); have.add(w)
            if len(chosen) >= n: break
    return chosen

fig, axes = plt.subplots(2, 4, figsize=(22, 10)); axes = axes.ravel()
for ax, lang in zip(axes, BENCH_LANGS):
    e = SERVICE._pick(lang); ws = pick_words(lang, e)
    X = np.stack([e.vec(w) for w, _ in ws])
    Y = TSNE(n_components=2, perplexity=min(30, (len(ws)-1)/3), init="pca", random_state=SEED).fit_transform(X)
    for one in (ax, None):
        if one is None: f2, one = plt.subplots(figsize=(8, 8))
        col = ["tab:blue" if s == "benchmark" else "tab:orange" for _, s in ws]
        one.scatter(Y[:, 0], Y[:, 1], c=col, s=18)
        for i, (x, y) in enumerate(Y): one.annotate(str(i+1), (x, y), fontsize=7, xytext=(2, 2), textcoords="offset points")
        one.set_title(f"{lang} — t-SNE of {len(ws)} words (blue: benchmark, orange: frequent)")
        if one is not ax: f2.tight_layout(); f2.savefig(OUT/"figures"/f"tsne_{lang}.png", dpi=150); plt.close(f2)
    pd.DataFrame({"index": range(1, len(ws)+1), "word": [w for w, _ in ws], "source": [s for _, s in ws]}).to_csv(
        OUT/"figures"/f"tsne_{lang}_words.csv", index=False, encoding="utf-8")
for ax in axes[len(BENCH_LANGS):]: ax.axis("off")
fig.tight_layout(); fig.savefig(OUT/"figures"/"tsne_all.png", dpi=110); plt.close(fig)
print("saved t-SNE figures + word tables to", OUT/"figures")

In [ ]:
# ----------------------------- analysis: good / bad predictions --------------
def rank01(x):
    r = pd.Series(x).rank(method="average").values
    return (r - 1) / max(1, len(r) - 1)
analysis = []
for lang in BENCH_LANGS:
    pairs = BENCH[lang]; e = SERVICE._pick(lang)
    rm, rh = rank01(SIMS[lang]), rank01([s for _, _, s in pairs])
    err = np.abs(rm - rh); order = np.argsort(err)
    pick = [("correct", i) for i in order[:3]] + [("incorrect", i) for i in order[::-1][:3]]
    for kind, i in pick:
        a, b, h = pairs[i]
        analysis.append(dict(language=lang, outcome=kind, word1=a, word2=b, human=h, cosine=round(SIMS[lang][i], 3),
                             rank_gap=round(float(err[i]), 3), oov=bool(e.is_oov(a) or e.is_oov(b))))
analysis = pd.DataFrame(analysis); analysis.to_csv(OUT/"results"/"analysis_examples.csv", index=False, encoding="utf-8")
print(analysis.to_string(index=False))

In [ ]:
# ----------------------------- disclosure + package --------------------------
DISCLOSURE = '''Pre-trained models used: NONE.   External corpora used: NONE (IndicCorpV2 only).
Libraries/tools: gensim FastText (training), HuggingFace `datasets` (loading IndicCorpV2), regex, numpy, scipy (Spearman),
scikit-learn (t-SNE), matplotlib, pandas.  Benchmark (Akhtar et al., 2017) human scores are NOT used for training;
they are used only for evaluation and to select among the pre-declared configurations in the experiments table.'''
open(OUT/"results"/"DISCLOSURE.txt", "w").write(DISCLOSURE); print(DISCLOSURE)

def make_zip(zip_name="FOVL_submission.zip", notebook="FOVL_embeddings.ipynb", report="report.pdf"):
    z = zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED)
    for d in ("embeddings", "vocab"):
        for p in sorted((OUT/d).iterdir()): z.write(p, f"{d}/{p.name}")
    for scope in ("mono", "multi"):                                    # trained models (for loading / OOV at test time)
        for p in sorted((MODELS/scope).rglob("*")):
            if p.is_file(): z.write(p, f"models/{scope}/{p.relative_to(MODELS/scope)}")
    z.write(MODELS/"config.json", "models/config.json")
    for p in ("final_spearman.csv", "oov_coverage.csv", "experiments.csv", "analysis_examples.csv", "DISCLOSURE.txt"):
        z.write(OUT/"results"/p, f"results/{p}")
    for p in sorted((OUT/"figures").iterdir()): z.write(p, f"figures/{p.name}")
    for extra_file in (notebook, report):
        if Path(extra_file).exists(): z.write(extra_file, Path(extra_file).name)
        else: print("NOTE: not found, not added ->", extra_file)
    z.close(); print(zip_name, round(Path(zip_name).stat().st_size / 1e6, 1), "MB")
# make_zip()    # run after the report PDF has been written to ./report.pdf